# GTEx model building with PCA, NFM and ICA with different number of LVs

💡 **Environment:** `clamp-analyses`  

## Libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pyprojroot.here import here

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, NMF, FastICA
import joblib

## Input

In [2]:
gtex_data = pd.read_csv(here("output/gtex/df_gtex_fbm_filt.csv"), index_col=0).astype(np.float32)

In [3]:
K = pd.read_csv(here("output/gtex/CLAMP_K_gtex.csv"), index_col=0)
n_components = int(K.shape[1])

## Output

In [4]:
out_dir = Path(here("output/gtex"))
out_dir.mkdir(parents=True, exist_ok=True)

# PCA

In [5]:
X = gtex_data.T  # samples x genes

pca_scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = pca_scaler.fit_transform(X)

pca = PCA(n_components=n_components, svd_solver="auto", random_state=0)
W = pca.fit_transform(X_scaled)   # samples x comps
H = pca.components_               # comps x genes

pc_names = [f"PC{i+1}" for i in range(W.shape[1])]

gtex_pca_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=pc_names)   # samples x comps
gtex_pca_loadings = pd.DataFrame(H, index=pc_names, columns=gtex_data.index)    # comps x genes
gtex_pca_B = gtex_pca_scores.T
gtex_pca_B.index.name = "PC"

gtex_pca_B.to_pickle(out_dir / "gtex_pca_B.pkl")
gtex_pca_scores.to_pickle(out_dir / "gtex_pca_scores.pkl")
gtex_pca_loadings.to_pickle(out_dir / "gtex_pca_loadings.pkl")

# ICA

In [6]:
X = gtex_data.T  # samples x genes

ica_scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = ica_scaler.fit_transform(X)

ica = FastICA(n_components=n_components, random_state=0, max_iter=2000)
W = ica.fit_transform(X_scaled)  # samples x comps
H = ica.mixing_.T                # comps x genes

ic_names = [f"IC{i+1}" for i in range(W.shape[1])]

gtex_ica_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=ic_names)   # samples x comps
gtex_ica_loadings = pd.DataFrame(H, index=ic_names, columns=gtex_data.index)    # comps x genes
gtex_ica_B = gtex_ica_scores.T
gtex_ica_B.index.name = "IC"

gtex_ica_B.to_pickle(out_dir / "gtex_ica_B.pkl")
gtex_ica_scores.to_pickle(out_dir / "gtex_ica_scores.pkl")
gtex_ica_loadings.to_pickle(out_dir / "gtex_ica_loadings.pkl")

# NMF

In [7]:
# non negative input need it
gene_min = gtex_data.min(axis=1)                 
gtex_data_nmf = gtex_data.sub(gene_min, axis=0) 

In [8]:
X = gtex_data_nmf.T  # samples x genes (non-negative)

nmf = NMF(n_components=n_components, init="nndsvd", random_state=0, max_iter=1000)
W = nmf.fit_transform(X)  # samples x comps
H = nmf.components_       # comps x genes

lv_names = [f"LV{i+1}" for i in range(W.shape[1])]

gtex_nmf_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=lv_names)  # samples x comps
gtex_nmf_loadings = pd.DataFrame(H, index=lv_names, columns=gtex_data.index)   # comps x genes
gtex_nmf_B = gtex_nmf_scores.T
gtex_nmf_B.index.name = "LV"

gtex_nmf_B.to_pickle(out_dir / "gtex_nmf_B.pkl")
gtex_nmf_scores.to_pickle(out_dir / "gtex_nmf_scores.pkl")
gtex_nmf_loadings.to_pickle(out_dir / "gtex_nmf_loadings.pkl")

/home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(
